# Growth Accounting Dashboard

This notebook reads `fct_growth_accounting` from BigQuery and visualizes daily, weekly, and monthly growth metrics.

In [ ]:
from google.cloud import bigquery
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ID = "your-gcp-project"
MART_DATASET = "mart"
TABLE = "fct_growth_accounting"

In [ ]:
client = bigquery.Client(project=PROJECT_ID)
query = f"""
select *
from `{PROJECT_ID}.{MART_DATASET}.{TABLE}`
order by period_grain, period_start
"""
df = client.query(query).to_dataframe()
df["period_start"] = pd.to_datetime(df["period_start"])
df.head()

In [ ]:
sns.set_theme(style="whitegrid")
for grain in ["daily", "weekly", "monthly"]:
    d = df[df["period_grain"] == grain].copy()
    if d.empty:
        continue

    plt.figure(figsize=(12, 5))
    plot_cols = ["active_users", "new_users", "retained_users", "resurrected_users", "churned_users"]
    melted = d.melt(id_vars=["period_start"], value_vars=plot_cols, var_name="metric", value_name="value")
    sns.lineplot(data=melted, x="period_start", y="value", hue="metric")
    plt.title(f"Growth Accounting ({grain})")
    plt.xlabel("Period")
    plt.ylabel("Users")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()